# 🛠️ Notebook 2: Cricinfo — Implementation

## 🛠️ Setup

```bash
cd 07-object-oriented-design/cricinfo
uv sync
```

Select the `.venv` kernel in VS Code (top-right). If it doesn't appear, reload the window: `Cmd+Shift+P` → **Reload Window**.


In [ ]:
from __future__ import annotations
from dataclasses import dataclass, field
from enum import Enum

class BallType(Enum):
    LEGAL = "legal"
    WIDE = "wide"       # +1 run, doesn't count as a legal ball
    NO_BALL = "no_ball" # +1 run, doesn't count as a legal ball

@dataclass
class Ball:
    runs: int = 0
    ball_type: BallType = BallType.LEGAL
    wicket: bool = False
    def total_runs(self) -> int:
        # wides/no-balls add 1 automatic run plus any runs taken
        extras = 1 if self.ball_type != BallType.LEGAL else 0
        return self.runs + extras
    def is_legal(self) -> bool:
        return self.ball_type == BallType.LEGAL

@dataclass
class Player:
    name: str

@dataclass
class Team:
    name: str
    players: list[Player] = field(default_factory=list)


In [ ]:
@dataclass
class Innings:
    batting: Team
    runs: int = 0
    wickets: int = 0
    legal_balls: int = 0           # track to compute overs
    log: list[Ball] = field(default_factory=list)

    def over_count(self) -> str:
        # cricket style: "12.4" = 12 overs and 4 balls
        return f"{self.legal_balls // 6}.{self.legal_balls % 6}"

    def record(self, ball: Ball) -> None:
        self.log.append(ball)
        self.runs += ball.total_runs()
        if ball.is_legal():
            self.legal_balls += 1
        if ball.wicket:
            self.wickets += 1

    def is_complete(self, max_overs: int = 20) -> bool:
        return self.wickets >= 10 or self.legal_balls >= max_overs * 6

    def summary(self) -> str:
        return f"{self.batting.name}: {self.runs}/{self.wickets} ({self.over_count()} ov)"


In [ ]:
@dataclass
class Match:
    team_a: Team
    team_b: Team
    max_overs: int = 20
    innings: list[Innings] = field(default_factory=list)

    def start(self, batting_first: Team):
        second = self.team_b if batting_first == self.team_a else self.team_a
        self.innings = [Innings(batting_first), Innings(second)]

    def current_innings(self) -> Innings:
        return next(i for i in self.innings if not i.is_complete(self.max_overs))

    def winner(self):
        if any(not i.is_complete(self.max_overs) for i in self.innings):
            return None
        a, b = self.innings
        if a.runs > b.runs: return a.batting.name
        if b.runs > a.runs: return b.batting.name
        return "Tie"

# demo
india = Team("India",  [Player(n) for n in ("Rohit","Kohli","Bumrah")])
aus   = Team("Aus",    [Player(n) for n in ("Warner","Smith","Cummins")])
m = Match(india, aus, max_overs=1)   # 1-over-per-side toy match
m.start(batting_first=india)

# India innings: 1,4,W,2,6,0  → 13/1
for b in [Ball(1), Ball(4), Ball(0, wicket=True), Ball(2), Ball(6), Ball(0)]:
    m.current_innings().record(b)
print(m.innings[0].summary())

# Aus chase: need 14. A wide gives +1 but doesn't count as a legal ball,
# so we bowl 6 legal balls to complete the innings.
for b in [Ball(0), Ball(0), Ball(1), Ball(1),
          Ball(runs=2, ball_type=BallType.WIDE), Ball(6), Ball(2)]:
    m.current_innings().record(b)
print(m.innings[1].summary())
print("Winner:", m.winner())


### Try it
- Add `Batter.runs` and `Batter.balls_faced` tracking with strike rotation.
- Add a `Bowler` whose bowling figures update per ball.
- Introduce `Commentary` as an Observer that gets notified on each ball.